# 🧪 Qwen2.5-0.5B GPU smoke on Kaggle

Import this notebook into Kaggle, select **GPU T4 x2**, enable internet access, and choose **Run All** in a fresh session. The code uses one T4 and downloads the pinned public model and Anthropic dataset.

The executable cell preserves the code used for the successful Kaggle run. It applies compatibility changes to a temporary checkout of `d23a49c` and uses Kaggle's CUDA build of PyTorch. The changes preserve sampled tokens between turns, use FP32 compute with 4-bit weights on T4, and accept standalone A/B answers or an exact, unique option text.

## Recorded result · November 9, 2026

| Check | Observed |
| --- | --- |
| Hardware | NVIDIA Tesla T4 |
| GRPO steps | 5 completed |
| Maximum absolute change in tracked LoRA weight | 5.72 × 10⁻⁶ |
| Saved adapter reload and inference | Passed |
| Adapter SHA-256 | `d5e7a6cb7e48068c78f0e38b44080c9580b7601eb112f377cbb31812a455e1c9` |

This smoke uses one pushback per example and skips validation generation. Measuring a change in sycophancy requires a matched held-out evaluation. On success, download `qwen05b-smoke-artifacts.zip` from Kaggle's Output panel.

In [ ]:
import importlib.metadata as metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

commit = 'd23a49c445af6571fa2a2b0aa883aff284a99ab4'
repo = Path('/kaggle/temp/sycophancy-rl-qwen05b')
output = Path('/kaggle/working/qwen05b-smoke-result')
output.mkdir(parents=True, exist_ok=True)
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'PYTHONUNBUFFERED': '1'}
python = sys.executable

def run(*args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(arg) for arg in args], cwd=cwd, env=env, check=True)

run('nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv')
run(python, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))')
repo.parent.mkdir(parents=True, exist_ok=True)
if not repo.exists():
    run('git', 'clone', 'https://github.com/kvelmurugan18/sycophancy-rl.git', repo)
run('git', 'checkout', '--detach', commit, cwd=repo)
rollout_path = repo / 'src/sycophancy_rl/training/online_rollout.py'
rollout_source = rollout_path.read_text(encoding='utf-8')
for old, new in [("                current_ids = _render_ids(tokenizer, current)\n                if current_ids != prompt_ids + completion_ids:\n                    raise RuntimeError(\n                        \"Chat-template history did not preserve the exact sampled token prefix; \"\n                        \"refusing misaligned multi-turn credit assignment.\"\n                    )", "                current_ids = prompt_ids + completion_ids"), ("                    next_ids = _render_ids(tokenizer, current)\n                    sampled_prefix = prompt_ids + completion_ids\n                    if next_ids[: len(sampled_prefix)] != sampled_prefix:\n                        raise RuntimeError(\n                            \"Decoded response could not be losslessly re-embedded by the chat template; \"\n                            \"refusing to attach the reward to different tokens.\"\n                        )\n                    environment_ids = next_ids[len(sampled_prefix) :]", "                    # Derive the template suffix without re-tokenizing the sampled answer.\n                    # This notebook uses exactly one pushback with the Qwen template.\n                    if environment_ranges or [m[\"role\"] for m in current[-2:]] != [\"assistant\", \"user\"]:\n                        raise RuntimeError(\"Unexpected conversation shape for the Qwen smoke.\")\n                    blank_history = [dict(message) for message in current]\n                    blank_history[-2][\"content\"] = \"\"\n                    blank_ids = _render_ids(tokenizer, blank_history)\n                    if blank_ids[: len(current_ids)] != current_ids:\n                        raise RuntimeError(\"Qwen template did not preserve the assistant header.\")\n                    environment_ids = blank_ids[len(current_ids) :]\n                    eos_id = tokenizer.eos_token_id\n                    if not environment_ids or environment_ids[0] != eos_id:\n                        raise RuntimeError(\"Qwen template suffix did not start with its turn terminator.\")\n                    if assistant_ids and assistant_ids[-1] == eos_id:\n                        environment_ids = environment_ids[1:]")]:
    if old in rollout_source:
        assert rollout_source.count(old) == 1
        rollout_source = rollout_source.replace(old, new)
    else:
        assert new in rollout_source
rollout_path.write_text(rollout_source, encoding='utf-8')
(output / 'qwen-token-preservation.patch').write_text(subprocess.check_output(['git', 'diff', '--', str(rollout_path.relative_to(repo))], cwd=repo, text=True), encoding='utf-8')
trainer_path = repo / 'src/sycophancy_rl/training/train_grpo.py'
trainer_source = trainer_path.read_text(encoding='utf-8')
trainer_source = trainer_source.replace('"torch_dtype": "auto",', '"torch_dtype": torch.float32,')
trainer_source = trainer_source.replace('compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16', 'compute_dtype = torch.float32')
trainer_source = trainer_source.replace('model.config.use_cache = False', 'model.config.use_cache = True')
trainer_path.write_text(trainer_source, encoding='utf-8')
(output / 'qwen-token-and-t4-compatibility.patch').write_text(subprocess.check_output(['git', 'diff', '--', 'src'], cwd=repo, text=True), encoding='utf-8')
environment_path = repo / 'src/sycophancy_rl/environment/online.py'
environment_source = environment_path.read_text(encoding='utf-8')
parse_original = '        parsed = parse_final_answer(model_response)'
parse_update = '''        parsed = parse_final_answer(model_response)
        if not parsed.valid and not parsed.contradictory:
            # Recognize exact choices while retaining the raw model response.
            choice = model_response.strip().rstrip(".!?").strip().casefold()
            labels = {label for label, option in self.trajectory.options.items()
                      if option.strip().rstrip(".!?").strip().casefold() == choice and choice}
            if choice in {"a", "b"}:
                labels = {choice.upper()}
            if len(labels) == 1:
                parsed = parse_final_answer(f"({next(iter(labels))})")'''
if parse_update not in environment_source:
    assert environment_source.count(parse_original) == 1
    environment_path.write_text(environment_source.replace(parse_original, parse_update), encoding='utf-8')
(output / 'smoke-compatibility.patch').write_text(subprocess.check_output(['git', 'diff', '--', 'src'], cwd=repo, text=True), encoding='utf-8')
torch_before = metadata.version('torch')
run(python, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '-r', 'deploy/kaggle/requirements.lock', cwd=repo)
run(python, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'tokenizers==0.22.2')
assert metadata.version('torch') == torch_before
print('Python:', sys.version.split()[0], '| running pinned source directly via PYTHONPATH')
env['PYTHONPATH'] = str(repo / 'src')
run(python, '-c', 'import torch; from peft import LoraConfig; from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig; from trl import GRPOTrainer', cwd=repo)
splits = repo / 'data/anthropic_experiment'
if not (splits / 'split_manifest.json').exists():
    run(python, '-m', 'sycophancy_rl.data_prep.prepare_anthropic_experiment', '--seed', '42', cwd=repo)
shutil.copy2(splits / 'split_manifest.json', output / 'split_manifest.json')
run_name = 'qwen05b-grpo-smoke'
print('Smoke-only configuration: skip validation generation; retain data split checks and all five GRPO steps.')
run(python, '-c', "from sycophancy_rl.cli.main import main\nfrom sycophancy_rl.training import train_grpo\n\noriginal = train_grpo.get_grpo_config\n\ndef smoke_config(**kwargs):\n    kwargs['overrides'] = {\n        **(kwargs.get('overrides') or {}),\n        'eval_strategy': 'no',\n        'eval_on_start': False,\n        'load_best_model_at_end': False,\n        'bf16': False,\n        'fp16': False,\n        'gradient_checkpointing': False,\n        'log_completions': False,\n    }\n    return original(**kwargs)\n\ntrain_grpo.get_grpo_config = smoke_config\nraise SystemExit(main())", 'train', '--run-name', run_name,
    '--profile', 'kaggle_online_smoke', '--model-id', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--train-path', splits / 'train.jsonl', '--validation-path', splits / 'validation.jsonl',
    '--benchmark-path', splits / 'benchmark.jsonl', '--output-root', output / 'checkpoints',
    '--rollout-mode', 'online', '--max-pushback-turns', '1', cwd=repo)
summary = json.loads((output / 'checkpoints' / run_name / 'training_summary.json').read_text())
assert summary['optimizer_update_verified']
assert summary['adapter_reload']['verified'] and summary['adapter_reload']['response'].strip()
status = json.loads((output / 'checkpoints' / run_name / 'run_status.json').read_text())
assert status['status'] == 'completed' and status['adapter_sha256']
print('GPU_SMOKE_PASSED', json.dumps({'source_commit': commit, **status, **summary}, indent=2))
print(shutil.make_archive('/kaggle/working/qwen05b-smoke-artifacts', 'zip', output))